# Smart Agriculture Data Analytics using Apache Spark

This notebook processes the India district-level Area, Production and Yield (APY) dataset (approximately 53.62 MiB as listed by India Data Portal). It performs data cleaning, crop/state/season/year aggregations, and production regression using Spark MLlib.

**Dataset source:** Ministry of Agriculture & Farmers Welfare, Government of India, via [India Data Portal](https://indiadataportal.com/p/area-production-yield-apy). Dataset coverage: 1997–2023, district-level. Download the CSV from the portal before running this notebook.

**Execution note:** This notebook uses Spark in Colab local mode; it is not a multi-node cluster deployment.

In [ ]:
!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless -qq
!pip -q install pyspark==3.5.1

import os, subprocess
java_home = subprocess.check_output('dirname $(dirname $(readlink -f $(which java)))', shell=True, text=True).strip()
os.environ['JAVA_HOME'] = java_home
print('JAVA_HOME:', java_home)
print('PySpark is ready')

In [ ]:
!git clone https://github.com/deecares/Smart-Agriculture-Crop-Recommendation.git
%cd Smart-Agriculture-Crop-Recommendation
!mkdir -p data

## Upload the large APY dataset

First download the CSV using the **Download Data** option on the [India Data Portal dataset page](https://indiadataportal.com/p/area-production-yield-apy). Then run the cell below and select the downloaded CSV file. Do not upload the old 2,200-row crop recommendation file.

In [ ]:
from google.colab import files
import os
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No file was uploaded.')
uploaded_name = next(iter(uploaded))
target_path = 'data/APY.csv'
os.replace(uploaded_name, target_path)
print('Saved:', target_path)
print('File size (MiB):', round(os.path.getsize(target_path) / (1024**2), 2))

In [ ]:
import pandas as pd
preview = pd.read_csv('data/APY.csv', nrows=5)
print('Columns:', list(preview.columns))
display(preview)

In [ ]:
!spark-submit src/agriculture_analytics.py --input data/APY.csv --output outputs

In [ ]:
import glob
metric_files = glob.glob('outputs/model_metrics/part-*.csv')
if not metric_files:
    raise FileNotFoundError('Model metrics were not created; check the Spark output above.')
metrics = pd.concat([pd.read_csv(p) for p in metric_files], ignore_index=True)
display(metrics.sort_values('rmse'))
print('Analytics output folders:', sorted(os.listdir('outputs')))